# 00 Data Processing


In [2]:
%load_ext autoreload
%autoreload 2

import pickle
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
from numpy import linalg as la

from config import (
    DATA_DIR,
    MOLECULE_DIR,
    PARAMETERS_DIR,
    RAW_MATRICES_DIR,
    PROCESSED_DATAFRAMES_DIR
)
from notebook_utils.general import read_npz

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### Statevector Dataset

In [3]:
SV_HAMILTONIAN_DIR = RAW_MATRICES_DIR / "SV_hamiltonian"
SV_SPIN_DIR = RAW_MATRICES_DIR / "SV_spin"
PYSCF_CASCI_DIR = RAW_MATRICES_DIR / "pyscf_casci"

def load_sv_hamiltonian_records(path):
    records = []

    for file in sorted(path.rglob("*.npz")):
        row = read_npz(file)
        H = row["H"]
        S = row["S"]

        records.append({
            "molecule": row["molecule"],
            "active_space": row["active_space"],
            "ansatz": row["ansatz"],
            "expansion": row["expansion"],
            "H": H,
            "S": S,
            "qse_dim": H.shape[0],
        })

    return pd.DataFrame(records)


def load_spin_records(path):
    records = []

    for file in sorted(path.rglob("*.npz")):
        row = read_npz(file)
        S2 = row["Z"]

        records.append({
            "molecule": row["molecule"],
            "active_space": row["active_space"],
            "ansatz": row["ansatz"],
            "expansion": row["expansion"],
            "S2": S2,
        })

    return pd.DataFrame(records)

def load_pyscf_casci_energy_records(path):
    records = []

    for file in sorted(path.rglob("*.npz")):
        row = read_npz(file)
        sectors = row["sectors"].tolist()

        if row["spin_type"] == "singlet":
            casci_energies = row["casci_energies"].tolist()
            casci_pvec = {
                str(root): vector
                for root, vector in enumerate(row["casci_pvec"])
            }
        else:
            casci_energies = [
                {sector: float(energy) for sector, energy in zip(sectors, nth_energies)}
                for nth_energies in row["casci_energies"]
            ]
            casci_pvec = {
                f"{root}_{sector}": vector
                for root, root_vectors in enumerate(row["casci_pvec"])
                for sector, vector in zip(sectors, root_vectors)
            }

        records.append({
            "molecule": row["molecule"],
            "active_space": row["active_space"],
            "spin_type": row["spin_type"],
            "pyscf_casci_energies": casci_energies,
            "pyscf_casci_pvec": casci_pvec,
        })

    return pd.DataFrame(records)


df_sv_hamiltonian = load_sv_hamiltonian_records(SV_HAMILTONIAN_DIR)
df_sv_spin = load_spin_records(SV_SPIN_DIR)
df_pyscf_casci = load_pyscf_casci_energy_records(PYSCF_CASCI_DIR)


In [5]:
# Compress the PYSCF Triplet energies
def average_triplet_sector_energies(casci_energies):
    average_energies = []
    sector_diffs = []

    for nth_energies in casci_energies:
        energies = list(nth_energies.values())
        average_energies.append(float(np.mean(energies)))
        sector_diffs.append(float(np.ptp(energies)))

    return average_energies, sector_diffs


new_casci_energies = []
triplet_sector_diffs = []

for _, row in df_pyscf_casci.iterrows():
    if row["spin_type"] == "triplet_all":
        energies, sector_diffs = average_triplet_sector_energies(
            row["pyscf_casci_energies"]
        )
        triplet_sector_diffs.extend(sector_diffs)
    else:
        energies = row["pyscf_casci_energies"]

    new_casci_energies.append(energies)


df_pyscf_casci["pyscf_casci_energies"] = new_casci_energies

max_triplet_sector_diff = max(triplet_sector_diffs, default=0.0)
print(f"Maximum PYSCF triplet sector energy difference: {max_triplet_sector_diff:.12e} Ha")


Maximum PYSCF triplet sector energy difference: 1.003218130791e-08 Ha


In [4]:
df_sv = df_sv_hamiltonian.merge(
    df_sv_spin,
    on=["molecule", "active_space", "ansatz", "expansion"],
    how="left",
)

df_sv = df_sv.merge(
    df_pyscf_casci,
    left_on=["molecule", "active_space", "expansion"],
    right_on=["molecule", "active_space", "spin_type"],
    how="left",
).drop(columns="spin_type")

df_sv = df_sv.sort_values(
    ["molecule", "active_space", "ansatz", "expansion"]
).reset_index(drop=True)

In [5]:
required_columns = ["H", "S", "S2", "pyscf_casci_energies", "pyscf_casci_pvec"]

df_sv_complete = df_sv.dropna(subset=required_columns).reset_index(drop=True)

PROCESSED_DATAFRAMES_DIR.mkdir(parents=True, exist_ok=True)
SV_DATAFRAME_PATH = PROCESSED_DATAFRAMES_DIR / "sv_qse_data.pkl"

df_sv_complete.to_pickle(SV_DATAFRAME_PATH)

df_sv_complete


,molecule,active_space,ansatz,expansion,H,S,qse_dim,S2,pyscf_casci_energies,pyscf_casci_pvec
0,Acetamide,2e2o,1UpCCGSDSinglet,singlet,"[[(-802.5633883162756+0j), (-4.698289217688655...","[[(3.9096898019756012+0j), (0.0228835930543396...",4,"[[(1.2656542480726785e-14+0j), 0j, (-6.5052130...","[-205.2944990546729, -204.82581813267245, -204...","{'0': [-0.15005468780119358, 0.0, -0.006284675..."
1,Acetamide,2e2o,1UpCCGSDSinglet,triplet,"[[(-0.00804123337029232+0j), 0j, 0j, (0.192703...","[[(3.9211269059064024e-05+0j), 0j, 0j, (-0.000...",12,"[[(7.842253811508881e-05+0j), 0j, 0j, (-0.0018...","[{'0': -205.07455033363988, 'p1': -205.0745503...","{'0_0': [1.2772011986346633e-16, 0.0, -0.70710..."
2,Acetamide,2e2o,2UpCCGSDSinglet,singlet,"[[(-802.5652804165668+0j), (-4.71721906907201+...","[[(3.9096990207381515+0j), (0.0229757934971892...",4,"[[(2.4868995751603507e-14+0j), (1.301042606982...","[-205.2944990546729, -204.82581813267245, -204...","{'0': [-0.15005468780119358, 0.0, -0.006284675..."
3,Acetamide,2e2o,2UpCCGSDSinglet,triplet,"[[(-0.008106060468175108+0j), 0j, 0j, (0.19346...","[[(3.9527383846466035e-05+0j), 0j, 0j, (-0.000...",12,"[[(7.905476768700626e-05+0j), 0j, 0j, (-0.0018...","[{'0': -205.07455033363988, 'p1': -205.0745503...","{'0_0': [1.2772011986346633e-16, 0.0, -0.70710..."
4,Acetamide,2e2o,3UpCCGSDSinglet,singlet,"[[(-802.5670885068249+0j), (-4.718526920088642...","[[(3.909707828949163+0j), (0.02298216377385717...",4,"[[(3.8219427622721014e-14+0j), 0j, (8.67361737...","[-205.2944990546729, -204.82581813267245, -204...","{'0': [-0.15005468780119358, 0.0, -0.006284675..."
...,...,...,...,...,...,...,...,...,...,...
1339,Uracil,4e4o,UCCGSD,triplet,"[[(-2.813670254929346+0j), 0j, 0j, (-0.4401863...","[[(0.006921185867571011+0j), 0j, 0j, (0.001083...",48,"[[(0.013842268554799125+0j), 0j, 0j, (0.002167...","[{'0': -406.93955355386265, 'p1': -406.9395535...","{'0_0': [-1.2972235967847874e-16, 0.0, -0.0716..."
1340,Uracil,4e4o,UCCSD,singlet,"[[(-1597.526898502948+0j), (5.5837888561038564...","[[(3.9241918014575696+0j), (-0.013719917786594...",16,"[[(4.233986869786599e-08+0j), (1.0771713044238...","[-407.1061847191804, -406.7961322794015, -406....","{'0': [-0.019559686447119955, 0.0, 0.000395949..."
1341,Uracil,4e4o,UCCSD,triplet,"[[(-2.813958608014113+0j), 0j, 0j, (-0.4301743...","[[(0.006921887370784002+0j), 0j, 0j, (0.001059...",48,"[[(0.013843763840662104+0j), 0j, 0j, (0.002118...","[{'0': -406.93955355386265, 'p1': -406.9395535...","{'0_0': [-1.2972235967847874e-16, 0.0, -0.0716..."
1342,Uracil,4e4o,UCCSDSinglet,singlet,"[[(-1597.5910915699835+0j), (5.547569853632924...","[[(3.924349500699329+0j), (-0.0136309133631264...",16,"[[(5.152821103593559e-08+0j), (1.7465805630121...","[-407.1061847191804, -406.7961322794015, -406....","{'0': [-0.019559686447119955, 0.0, 0.000395949..."


### Shots dataset

In [6]:
SHOTS_HAMILTONIAN_DIR = RAW_MATRICES_DIR / "shots_hamiltonian"


def load_shots_hamiltonian_records(path):
    records = []

    for file in sorted(path.rglob("*.npz")):
        row = read_npz(file)
        h_keys = sorted(key for key in row if key.startswith("H_shots_"))

        for h_key in h_keys:
            sample_key = h_key.removeprefix("H_")
            _, n_shots, repeat = sample_key.split("_")
            H = row[f"H_{sample_key}"]
            S = row[f"S_{sample_key}"]

            records.append({
                "molecule": row["molecule"],
                "active_space": row["active_space"],
                "ansatz": row["ansatz"],
                "expansion": row["expansion"],
                "sample_key": sample_key,
                "n_shots": int(n_shots),
                "repeat": int(repeat),
                "H_shots": H,
                "S_shots": S,
                "qse_dim": H.shape[0],
                "num_commuting_groups": int(row["num_commuting_groups"]),
            })

    return pd.DataFrame(records)


df_shots_hamiltonian = load_shots_hamiltonian_records(SHOTS_HAMILTONIAN_DIR)

df_shots_sv = df_sv_hamiltonian.rename(columns={
    "H": "H_sv",
    "S": "S_sv",
    "qse_dim": "qse_dim_sv",
})

df_shots = df_shots_hamiltonian.merge(
    df_shots_sv,
    on=["molecule", "active_space", "ansatz", "expansion"],
    how="left",
)

df_shots = df_shots.sort_values(
    ["molecule", "active_space", "ansatz", "expansion", "n_shots", "repeat"]
).reset_index(drop=True)


In [7]:
required_columns = ["H_shots", "S_shots", "H_sv", "S_sv"]

df_shots_complete = df_shots.dropna(subset=required_columns).reset_index(drop=True)

PROCESSED_DATAFRAMES_DIR.mkdir(parents=True, exist_ok=True)
SHOTS_DATAFRAME_PATH = PROCESSED_DATAFRAMES_DIR / "shots_qse_data.pkl"

df_shots_complete.to_pickle(SHOTS_DATAFRAME_PATH)

df_shots_complete


,molecule,active_space,ansatz,expansion,sample_key,n_shots,repeat,H_shots,S_shots,qse_dim,num_commuting_groups,H_sv,S_sv,qse_dim_sv
0,Acetamide,2e2o,UCCSD,singlet,shots_1000_0,1000,0,"[[(-799.3416862999425+0j), (-9.698487335464566...","[[(3.894+0j), (0.047250000000000014+0.02125000...",4,25,"[[(-802.566868206377+0j), (-4.71630877716983+0...","[[(3.9097067555916256+0j), (0.0229713600038418...",4
1,Acetamide,2e2o,UCCSD,singlet,shots_1000_1,1000,1,"[[(-798.521323691679+0j), (-5.749462200016003+...","[[(3.8899999999999997+0j), (0.0280000000000000...",4,25,"[[(-802.566868206377+0j), (-4.71630877716983+0...","[[(3.9097067555916256+0j), (0.0229713600038418...",4
2,Acetamide,2e2o,UCCSD,singlet,shots_1000_2,1000,2,"[[(-799.7532554896775+0j), (-5.648868316573327...","[[(3.896+0j), (0.027499999999999997-0.00400000...",4,25,"[[(-802.566868206377+0j), (-4.71630877716983+0...","[[(3.9097067555916256+0j), (0.0229713600038418...",4
3,Acetamide,2e2o,UCCSD,singlet,shots_1000_3,1000,3,"[[(-804.2721954287647+0j), (-7.493082463093295...","[[(3.918+0j), (0.03650000000000003+0.021000000...",4,25,"[[(-802.566868206377+0j), (-4.71630877716983+0...","[[(3.9097067555916256+0j), (0.0229713600038418...",4
4,Acetamide,2e2o,UCCSD,singlet,shots_1000_4,1000,4,"[[(-799.3431587075672+0j), (3.7451390568854235...","[[(3.894+0j), (-0.01825000000000003-0.03225000...",4,25,"[[(-802.566868206377+0j), (-4.71630877716983+0...","[[(3.9097067555916256+0j), (0.0229713600038418...",4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22395,Uracil,4e4o,UCCSD,triplet,shots_1000000_5,1000000,5,"[[(-2.8145809704301428+0j), (-0.02413673592364...","[[(0.006923500000000027+0j), (5.97505230102345...",48,2952,"[[(-2.813958608014113+0j), 0j, 0j, (-0.4301743...","[[(0.006921887370784002+0j), 0j, 0j, (0.001059...",48
22396,Uracil,4e4o,UCCSD,triplet,shots_1000000_6,1000000,6,"[[(-2.8272278884620192+0j), (-0.02591627631869...","[[(0.006954499999999975+0j), (6.36396103068114...",48,2952,"[[(-2.813958608014113+0j), 0j, 0j, (-0.4301743...","[[(0.006921887370784002+0j), 0j, 0j, (0.001059...",48
22397,Uracil,4e4o,UCCSD,triplet,shots_1000000_7,1000000,7,"[[(-2.807809917760352+0j), (-0.128365138962330...","[[(0.00690650000000001+0j), (0.000316076731190...",48,2952,"[[(-2.813958608014113+0j), 0j, 0j, (-0.4301743...","[[(0.006921887370784002+0j), 0j, 0j, (0.001059...",48
22398,Uracil,4e4o,UCCSD,triplet,shots_1000000_8,1000000,8,"[[(-2.844005022828796+0j), (0.0115844943642761...","[[(0.006996000000000002+0j), (-2.8284271247468...",48,2952,"[[(-2.813958608014113+0j), 0j, 0j, (-0.4301743...","[[(0.006921887370784002+0j), 0j, 0j, (0.001059...",48
